In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')


In [3]:
df = pd.read_csv('Dataset .csv', encoding='utf-8')
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
df.info()
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   str    
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   Locality Verbose      9551 non-null   str    
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   str    
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   str    
 12  Has Table booking     9551 non-null   str    
 13  Has Online delivery   9551 non-null   str    
 14  Is delivering now     9551 non-null   str    
 15  Switch to order menu  9551 non-n

In [5]:
cols = ['Restaurant Name', 'Cuisines', 'Price range', 'Average Cost for two', 
        'City', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 
        'Aggregate rating', 'Votes']
df_rec = df[cols].copy()
df_rec.head()

,Restaurant Name,Cuisines,Price range,Average Cost for two,City,Has Table booking,Has Online delivery,Is delivering now,Aggregate rating,Votes
0,Le Petit Souffle,"French, Japanese, Desserts",3,1100,Makati City,Yes,No,No,4.8,314
1,Izakaya Kikufuji,Japanese,3,1200,Makati City,Yes,No,No,4.5,591
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4000,Mandaluyong City,Yes,No,No,4.4,270
3,Ooma,"Japanese, Sushi",4,1500,Mandaluyong City,No,No,No,4.9,365
4,Sambo Kojin,"Japanese, Korean",4,1500,Mandaluyong City,Yes,No,No,4.8,229


In [6]:
print(df_rec.isnull().sum())
df_rec = df_rec.dropna(subset=['Cuisines']).reset_index(drop=True)
print(df_rec.isnull().sum())
df_rec = df_rec.dropna(subset=['Price range', 'Average Cost for two']).reset_index(drop=True)
print("Shape after dropping missing:", df_rec.shape)

Restaurant Name         0
Cuisines                9
Price range             0
Average Cost for two    0
City                    0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Aggregate rating        0
Votes                   0
dtype: int64
Restaurant Name         0
Cuisines                0
Price range             0
Average Cost for two    0
City                    0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Aggregate rating        0
Votes                   0
dtype: int64
Shape after dropping missing: (9542, 10)


In [7]:
df_rec['Cuisines_clean'] = df_rec['Cuisines'].apply(lambda x: ' '.join(x.split(', ')))
tfidf = TfidfVectorizer(max_features=100)
cuisine_matrix = tfidf.fit_transform(df_rec['Cuisines_clean'])
print("Cuisine matrix shape:", cuisine_matrix.shape)

Cuisine matrix shape: (9542, 100)


In [8]:
top_cities = df_rec['City'].value_counts().head(20).index
df_rec['City_top'] = df_rec['City'].apply(lambda x: x if x in top_cities else 'Other')
city_dummies = pd.get_dummies(df_rec['City_top'], prefix='City')
df_rec['Has Table booking'] = df_rec['Has Table booking'].map({'Yes':1, 'No':0})
df_rec['Has Online delivery'] = df_rec['Has Online delivery'].map({'Yes':1, 'No':0})
df_rec['Is delivering now'] = df_rec['Is delivering now'].map({'Yes':1, 'No':0})
numeric_features = ['Price range', 'Average Cost for two']

In [9]:
scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(df_rec[numeric_features])
scaled_numeric_sparse = csr_matrix(scaled_numeric)
binary_sparse = csr_matrix(df_rec[['Has Table booking', 'Has Online delivery', 'Is delivering now']].values)
city_sparse = csr_matrix(city_dummies.values)
combined_features = hstack([cuisine_matrix, scaled_numeric_sparse, binary_sparse, city_sparse])
print("Combined feature matrix shape:", combined_features.shape)

Combined feature matrix shape: (9542, 126)


In [10]:
nn_model = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
nn_model.fit(combined_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [11]:
def recommend_restaurants_by_index(restaurant_idx, n_recommendations=5):
    distances, indices = nn_model.kneighbors(combined_features[restaurant_idx], n_neighbors=n_recommendations+1)
    rec_indices = indices.flatten()[1:n_recommendations+1]
    rec_distances = distances.flatten()[1:n_recommendations+1]
    recommendations = df_rec.iloc[rec_indices][['Restaurant Name', 'Cuisines', 'Price range', 'Average Cost for two', 'City', 'Aggregate rating']].copy()
    recommendations['Similarity Score'] = 1 - rec_distances
    return recommendations

sample_idx = 0
print("Query restaurant:")
print(df_rec.iloc[sample_idx][['Restaurant Name', 'Cuisines', 'City', 'Price range', 'Average Cost for two']])
print("\nRecommendations:")
print(recommend_restaurants_by_index(sample_idx, n_recommendations=5))

Query restaurant:
Restaurant Name                   Le Petit Souffle
Cuisines                French, Japanese, Desserts
City                                   Makati City
Price range                                      3
Average Cost for two                          1100
Name: 0, dtype: object

Recommendations:
                                        Restaurant Name  \
1                                      Izakaya Kikufuji   
2297  Jonathan's Kitchen - Holiday Inn Express & Suites   
9382                                               Nobu   
4                                           Sambo Kojin   
20                                       NIU by Vikings   

                                        Cuisines  Price range  \
1                                       Japanese            3   
2297             North Indian, Japanese, Italian            3   
9382                             Japanese, Sushi            4   
4                               Japanese, Korean            4   
20    

In [12]:
def recommend_by_preferences(preferred_cuisines, max_price_range=None, max_cost=None, city=None, n_recommendations=5):
    cuisine_str = ' '.join(preferred_cuisines)
    cuisine_vec = tfidf.transform([cuisine_str])
    price = max_price_range if max_price_range is not None else df_rec['Price range'].median()
    cost = max_cost if max_cost is not None else df_rec['Average Cost for two'].median()
    numeric_query = np.array([[price, cost]])
    numeric_query_scaled = scaler.transform(numeric_query)
    numeric_sparse = csr_matrix(numeric_query_scaled)
    binary_query = csr_matrix(np.array([[0, 0, 0]]))
    city_cols = city_dummies.columns
    city_query = np.zeros((1, len(city_cols)))
    if city in top_cities:
        city_query[0, city_cols.get_loc(f'City_{city}')] = 1
    else:
        city_query[0, city_cols.get_loc('City_Other')] = 1
    city_sparse_query = csr_matrix(city_query)
    query_vector = hstack([cuisine_vec, numeric_sparse, binary_query, city_sparse_query])
    distances, indices = nn_model.kneighbors(query_vector, n_neighbors=n_recommendations)
    rec_indices = indices.flatten()
    rec_distances = distances.flatten()
    recommendations = df_rec.iloc[rec_indices][['Restaurant Name', 'Cuisines', 'Price range', 'Average Cost for two', 'City', 'Aggregate rating']].copy()
    recommendations['Similarity Score'] = 1 - rec_distances
    return recommendations

prefs = {
    'preferred_cuisines': ['North Indian', 'Chinese'],
    'max_price_range': 3,
    'max_cost': 1500,
    'city': 'New Delhi'
}
recs = recommend_by_preferences(**prefs, n_recommendations=5)
print("Recommendations based on preferences:")
print(recs)


Recommendations based on preferences:
      Restaurant Name               Cuisines  Price range  \
4272  Barbeque Nation  North Indian, Chinese            3   
3086  Barbeque Nation  North Indian, Chinese            3   
5886  Barbeque Nation  North Indian, Chinese            3   
3632  Barbeque Nation  North Indian, Chinese            3   
6986          Turn Up  North Indian, Chinese            3   

      Average Cost for two       City  Aggregate rating  Similarity Score  
4272                  1600  New Delhi               4.0          0.999995  
3086                  1600  New Delhi               4.1          0.999995  
5886                  1600  New Delhi               4.1          0.999995  
3632                  1600  New Delhi               4.0          0.999995  
6986                  1200  New Delhi               3.0          0.999954  


In [13]:
def recommend_by_preferences_sorted(preferred_cuisines, max_price_range=None, max_cost=None, city=None, n_recommendations=5):
    recs = recommend_by_preferences(preferred_cuisines, max_price_range, max_cost, city, n_recommendations*3)
    if max_price_range is not None:
        recs = recs[recs['Price range'] <= max_price_range]
    if max_cost is not None:
        recs = recs[recs['Average Cost for two'] <= max_cost]
    recs = recs.sort_values('Aggregate rating', ascending=False).head(n_recommendations)
    return recs

recs_sorted = recommend_by_preferences_sorted(**prefs, n_recommendations=5)
print("Recommendations sorted by rating:")
print(recs_sorted)

Recommendations sorted by rating:
          Restaurant Name                        Cuisines  Price range  \
6647         Kopper Kadai                    North Indian            3   
6708      The Drunk House  North Indian, Italian, Chinese            3   
6661                Dhaba           Chinese, North Indian            3   
3046  My Bar Headquarters  North Indian, Mughlai, Chinese            3   
6627        Gola Sizzlers  Mughlai, North Indian, Chinese            3   

      Average Cost for two       City  Aggregate rating  Similarity Score  
6647                  1400  New Delhi               4.8          0.935021  
6708                  1300  New Delhi               4.2          0.919496  
6661                  1100  New Delhi               3.7          0.999918  
3046                  1000  New Delhi               3.7          0.926655  
6627                  1300  New Delhi               3.4          0.926761  


In [14]:
test_restaurant = df_rec[df_rec['Restaurant Name'] == 'Le Petit Souffle'].iloc[0]
print("Test restaurant:")
print(test_restaurant[['Restaurant Name', 'Cuisines', 'City', 'Price range', 'Average Cost for two']])
idx = test_restaurant.name
recs = recommend_restaurants_by_index(idx, n_recommendations=5)
print("\nSimilar restaurants:")
print(recs)

# %% Cell 14: Save Model
model_artifacts = {
    'tfidf': tfidf,
    'scaler': scaler,
    'nn_model': nn_model,
    'top_cities': top_cities,
    'city_columns': city_dummies.columns.tolist(),
    'feature_names': ['cuisine_tfidf', 'numeric', 'binary', 'city']
}
joblib.dump(model_artifacts, 'restaurant_recommender.pkl')
print("Model artifacts saved.")

Test restaurant:
Restaurant Name                   Le Petit Souffle
Cuisines                French, Japanese, Desserts
City                                   Makati City
Price range                                      3
Average Cost for two                          1100
Name: 0, dtype: object

Similar restaurants:
                                        Restaurant Name  \
1                                      Izakaya Kikufuji   
2297  Jonathan's Kitchen - Holiday Inn Express & Suites   
9382                                               Nobu   
4                                           Sambo Kojin   
20                                       NIU by Vikings   

                                        Cuisines  Price range  \
1                                       Japanese            3   
2297             North Indian, Japanese, Italian            3   
9382                             Japanese, Sushi            4   
4                               Japanese, Korean            4   
20 

In [15]:
loaded = joblib.load('restaurant_recommender.pkl')
tfidf_loaded = loaded['tfidf']
scaler_loaded = loaded['scaler']
nn_model_loaded = loaded['nn_model']
top_cities_loaded = loaded['top_cities']
city_columns_loaded = loaded['city_columns']

def recommend_loaded(preferred_cuisines, max_price_range=None, max_cost=None, city=None, n_recommendations=5):
    cuisine_str = ' '.join(preferred_cuisines)
    cuisine_vec = tfidf_loaded.transform([cuisine_str])
    price = max_price_range if max_price_range is not None else df_rec['Price range'].median()
    cost = max_cost if max_cost is not None else df_rec['Average Cost for two'].median()
    numeric_query = np.array([[price, cost]])
    numeric_query_scaled = scaler_loaded.transform(numeric_query)
    numeric_sparse = csr_matrix(numeric_query_scaled)
    binary_query = csr_matrix(np.array([[0, 0, 0]]))
    city_query = np.zeros((1, len(city_columns_loaded)))
    if city in top_cities_loaded:
        city_query[0, city_columns_loaded.index(f'City_{city}')] = 1
    else:
        city_query[0, city_columns_loaded.index('City_Other')] = 1
    city_sparse_query = csr_matrix(city_query)
    query_vector = hstack([cuisine_vec, numeric_sparse, binary_query, city_sparse_query])
    distances, indices = nn_model_loaded.kneighbors(query_vector, n_neighbors=n_recommendations)
    rec_indices = indices.flatten()
    rec_distances = distances.flatten()
    recommendations = df_rec.iloc[rec_indices][['Restaurant Name', 'Cuisines', 'Price range', 'Average Cost for two', 'City', 'Aggregate rating']].copy()
    recommendations['Similarity Score'] = 1 - rec_distances
    return recommendations

print(recommend_loaded(['Italian', 'Pizza'], max_price_range=3, max_cost=1000, city='New Delhi'))


           Restaurant Name                        Cuisines  Price range  \
5388             Pizza Hut       Italian, Pizza, Fast Food            3   
3651            Baking Bad                  Pizza, Italian            3   
7061            Fat Lulu's                  Pizza, Italian            3   
3981  Moonshine Cafe & Bar            Continental, Italian            4   
5410       Moets Curry Pot  North Indian, Chinese, Italian            3   

      Average Cost for two       City  Aggregate rating  Similarity Score  
5388                  1000  New Delhi               2.8          0.959368  
3651                  1000  New Delhi               3.8          0.888319  
7061                  1250  New Delhi               3.8          0.888287  
3981                  2400  New Delhi               3.5          0.856332  
5410                  1200  New Delhi               0.0          0.855634  
